# Ejercicio 6: Actividad Integradora: Del ADN a la Proteína

## 📖 Introducción y Fundamento Teórico Resumido

> **Objetivo:** Recorrer el Dogma Central de la Biología Molecular completo (Replicación $\to$ Transcripción $\to$ Traducción) integrando los procesos en un pipeline bioinformático continuo, transparente y monitorizado paso a paso.

### Resumen de los puntos teóricos del enunciado:

1. **Secuencia biológica de partida (Banco público NCBI):**
   - Gen de la **Insulina humana (*INS*)**, cromosoma $11\text{p}15.5$.
   - Región codificante (CDS) de referencia: **NCBI RefSeq `NM_000207.3`** / Proteína **`NP_000198.1`**.
   - Longitud: **$333$ pares de bases (pb)** ($111$ codones: 1 inicio `ATG`, 109 con sentido, 1 parada `TAG`).

2. **Paso 1: Replicación Semiconservativa del ADN:**
   - La **ADN Helicasa** abre la doble hélice y la **ADN Polimerasa** polimeriza en sentido $5' \to 3'$.
   - Al término de 1 ronda, surgen **2 moléculas hijas bicatenarias idénticas**, cada una compuesta por una hebra parental original y una hebra neosintetizada complementaria.

3. **Paso 2: Transcripción a ARN Mensajero (ARNm):**
   - La **ARN Polimerasa II** lee la hebra molde en sentido $3' \to 5'$ y sintetiza el ARNm en sentido $5' \to 3'$.
   - Se obtiene un transcrito maduro de **$333$ ribonucleótidos** donde cada Timina ($T$) es reemplazada por Uracilo ($U$).

4. **Paso 3: Traducción ribosómica a Proteína:**
   - El ribosoma traduce los 111 codones en la **Preproinsulina humana (110 aminoácidos)**, estructurada en 4 dominios esenciales:
     - **Péptido Señal (aa 1-24):** Dirige la proteína al retículo endoplásmico.
     - **Cadena B (aa 25-54):** Primera cadena de la hormona madura activa.
     - **Péptido C conector (aa 55-89):** Guía el plegamiento y la formación de los 3 puentes disulfuro; luego es escindido.
     - **Cadena A (aa 90-110):** Segunda cadena de la hormona madura activa.

5. **Reflexión: ¿Qué punto del proceso es más vulnerable a errores?**
   - **Tasa de error basal:** Traducción ($\sim 10^{-4}$) $>$ Transcripción ($\sim 10^{-5}$) $>$ Replicación ($\sim 10^{-9}-10^{-10}$).
   - **Vulnerabilidad e impacto cualitativo:** A pesar de su altísima fidelidad, **la Replicación del ADN es el punto más vulnerable**:
     - Un error de traducción o transcripción es **transitorio y efímero** (el ARNm y las proteínas erróneas se degradan rápidamente sin afectar al resto de copias normales).
     - Un error de replicación no corregido se fija como una **mutación permanente en el genoma**, propagándose al **$100\%$ de los ARNm futuros y al $100\%$ de las proteínas sintetizadas**, transmitiéndose clonalmente (cáncer) o generacionalmente (enfermedades hereditarias).

# 💻 Extensión con Biopython: Pipeline Bioinformático del Dogma Central

> **Enunciado de la extensión:**
> *Programa un pipeline que realice los tres pasos: generar la hebra complementaria (replicación), obtener el transcrito (transcripción) y traducir a proteína (traducción), informando en todo momento de lo que está ocurriendo en los procesos.*

A continuación, construiremos y ejecutaremos el pipeline paso a paso con monitorización continua.

### Paso 0: Importación de módulos y localización de archivos FASTA

Configuramos las rutas a los archivos FASTA de entrada (`data/insulina_humana_cds.fasta`) y de salida para los transcritos de ARNm y proteína generados por el pipeline.

In [1]:
from pathlib import Path
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq

# Localización robusta tanto si ejecutamos desde notebooks/ como desde la raíz
base_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
fasta_dna_input = base_dir / "data" / "insulina_humana_cds.fasta"
fasta_mrna_output = base_dir / "data" / "ejercicio_06_insulina_arnm.fasta"
fasta_prot_output = base_dir / "data" / "ejercicio_06_insulina_proteina.fasta"

print(f"Directorio de trabajo:  {base_dir}")
print(f"Archivo FASTA entrada:  {fasta_dna_input.name} (Existe: {fasta_dna_input.exists()})")

Directorio de trabajo:  C:\Users\Ayoze\Desktop\Del-ADN-a-la-protenia
Archivo FASTA entrada:  insulina_humana_cds.fasta (Existe: True)


### Paso 1: Carga e Inspección de la CDS del gen de la Insulina con `Bio.SeqIO`

Cargamos la secuencia codificante con `SeqIO.read()`, inspeccionamos sus metadatos y verificamos que cumpla las condiciones canónicas de un marco abierto de lectura (ORF): inicio en `ATG`, longitud múltiplo de 3 y terminación en codón de parada (`TAG`).

In [2]:
# Lectura de la secuencia de ADN con Biopython
record_dna = SeqIO.read(fasta_dna_input, "fasta")
dna_seq = record_dna.seq

print("=" * 75)
print("PASO 0: CARGA E INSPECCIÓN DE LA SECUENCIA DE ADN (NCBI GenBank)")
print("=" * 75)
print(f"ID de registro:         {record_dna.id}")
print(f"Descripción:            {record_dna.description}")
print(f"Longitud total:         {len(dna_seq)} pares de bases (pb)")
print(f"Número de codones:      {len(dna_seq) // 3} tripletes")
print(f"Extremo 5' inicial:     5'- {dna_seq[:36]}... -3'")
print(f"Extremo 3' terminal:    ...{dna_seq[-36:]} -3'")

# Verificaciones del marco abierto de lectura
assert len(dna_seq) % 3 == 0, "Error: La longitud no es múltiplo de 3"
assert str(dna_seq).startswith("ATG"), "Error: No comienza con el codón de inicio ATG"
assert str(dna_seq).endswith(("TAG", "TAA", "TGA")), "Error: No termina con codón de parada"
print("\n[VERIFICACIÓN] Marco abierto de lectura (ORF) canónico confirmado: Inicio ATG y Fin TAG.")
print("=" * 75)

PASO 0: CARGA E INSPECCIÓN DE LA SECUENCIA DE ADN (NCBI GenBank)
ID de registro:         lcl|NM_000207.3_cds_NP_000198.1_1
Descripción:            lcl|NM_000207.3_cds_NP_000198.1_1 [gene=INS] [db_xref=CCDS:CCDS7729.1] [protein=insulin preproprotein] [protein_id=NP_000198.1] [location=60..392] [gbkey=CDS]
Longitud total:         333 pares de bases (pb)
Número de codones:      111 tripletes
Extremo 5' inicial:     5'- ATGGCCCTGTGGATGCGCCTCCTGCCCCTGCTGGCG... -3'
Extremo 3' terminal:    ...TGCTCCCTCTACCAGCTGGAGAACTACTGCAACTAG -3'

[VERIFICACIÓN] Marco abierto de lectura (ORF) canónico confirmado: Inicio ATG y Fin TAG.


### Paso 2: Replicación Semiconservativa del ADN con Biopython

Simulamos el proceso de replicación:
1. La **ADN Helicasa** abre el dúplex parental.
2. Obtenemos la **hebra complementaria directa** ($3' \to 5'$) con `dna_seq.complement()` (que actúa como molde físico).
3. Obtenemos la **reversa complementaria** ($5' \to 3'$) con `dna_seq.reverse_complement()`.
4. Esquematizamos las dos moléculas hijas bicatenarias resultantes tras una ronda de replicación.

In [3]:
# 1. Replicación semiconservativa
dna_template_3_5 = dna_seq.complement()
dna_rev_comp_5_3 = dna_seq.reverse_complement()

print("=" * 75)
print("PASO 1: REPLICACIÓN SEMICONSERVATIVA DEL ADN")
print("=" * 75)
print("[MECANISMO MOLECULAR]")
print("  * La ADN Helicasa desnaturaliza el dúplex parental rompiendo puentes de H.")
print("  * La Primasa sintetiza iniciadores cortos de ARN.")
print("  * La ADN Polimerasa polimeriza dNTPs en sentido 5' -> 3' con autocorrección 3' -> 5'.")

print("\n[SECUENCIAS GENERADAS]")
print(f"  > Hebra parental (codificante 5'->3'):   5'- {dna_seq[:30]}... -3'")
print(f"  > Hebra complementaria (molde 3'->5'):  3'- {dna_template_3_5[:30]}... -5'")
print(f"  > Hebra complementaria (reversa 5'->3'): 5'- {dna_rev_comp_5_3[:30]}... -3'")

print("\n[MOLÉCULAS HIJAS BICATENARIAS TRAS LA RONDA 1]")
print("  +-------------------------------------------------------------+")
print("  | Molécula Hija 1:                                            |")
print(f"  |   [Parental]     5'- {dna_seq[:28]}... -3'            |")
print(f"  |   [Neosíntesis]  3'- {dna_template_3_5[:28]}... -5'            |")
print("  +-------------------------------------------------------------+")
print("  | Molécula Hija 2:                                            |")
print(f"  |   [Neosíntesis]  5'- {dna_seq[:28]}... -3'            |")
print(f"  |   [Parental]     3'- {dna_template_3_5[:28]}... -5'            |")
print("  +-------------------------------------------------------------+")
print("[ESTADO] Replicación completada con 100% de fidelidad de bases.")
print("=" * 75)

PASO 1: REPLICACIÓN SEMICONSERVATIVA DEL ADN
[MECANISMO MOLECULAR]
  * La ADN Helicasa desnaturaliza el dúplex parental rompiendo puentes de H.
  * La Primasa sintetiza iniciadores cortos de ARN.
  * La ADN Polimerasa polimeriza dNTPs en sentido 5' -> 3' con autocorrección 3' -> 5'.

[SECUENCIAS GENERADAS]
  > Hebra parental (codificante 5'->3'):   5'- ATGGCCCTGTGGATGCGCCTCCTGCCCCTG... -3'
  > Hebra complementaria (molde 3'->5'):  3'- TACCGGGACACCTACGCGGAGGACGGGGAC... -5'
  > Hebra complementaria (reversa 5'->3'): 5'- CTAGTTGCAGTAGTTCTCCAGCTGGTAGAG... -3'

[MOLÉCULAS HIJAS BICATENARIAS TRAS LA RONDA 1]
  +-------------------------------------------------------------+
  | Molécula Hija 1:                                            |
  |   [Parental]     5'- ATGGCCCTGTGGATGCGCCTCCTGCCCC... -3'            |
  |   [Neosíntesis]  3'- TACCGGGACACCTACGCGGAGGACGGGG... -5'            |
  +-------------------------------------------------------------+
  | Molécula Hija 2:                        

### Paso 3: Transcripción a ARN Mensajero (ARNm) con `.transcribe()`

Simulamos la acción de la **ARN Polimerasa II**:
- La enzima lee la hebra molde antiparalela ($3' \to 5'$) y polimeriza ribonucleótidos en sentido $5' \to 3'$.
- `dna_seq.transcribe()` genera el transcrito de ARNm maduro reemplazando las timinas por uracilos.
- Validamos que no queden residuos de Timina y exportamos el resultado a `data/ejercicio_06_insulina_arnm.fasta` con `SeqIO.write()`.

In [4]:
# 2. Transcripción de ADN a ARNm
mrna_seq = dna_seq.transcribe()

print("=" * 75)
print("PASO 2: TRANSCRIPCIÓN DE ADN A ARN MENSAJERO (ARNm)")
print("=" * 75)
print("[MECANISMO MOLECULAR]")
print("  * La ARN Polimerasa II reconoce el promotor y se acopla a la hebra molde (3'->5').")
print("  * Incorpora NTPs complementarios (A->U, T->A, C->G, G->C) en sentido 5' -> 3'.")

print("\n[SECUENCIA TRANSCRITA]")
print(f"  > Molde de ADN leído (3'->5'):  3'- {dna_template_3_5[:30]}... -5'")
print(f"  > Transcrito de ARNm (5'->3'):  5'- {mrna_seq[:30]}... -3'")
print(f"  > Longitud del ARNm:            {len(mrna_seq)} ribonucleótidos")

# Validación bioquímica
assert "T" not in str(mrna_seq), "Error: Se encontraron residuos de Timina en el ARN"
assert str(mrna_seq).startswith("AUG"), "Error: El ARNm no inicia en AUG"
assert str(mrna_seq).endswith("UAG"), "Error: El ARNm no finaliza en UAG"

# Exportar archivo FASTA de ARNm
mrna_record = SeqRecord(
    mrna_seq,
    id=f"{record_dna.id}_mRNA",
    description="ARNm de Insulina humana generado con el pipeline Biopython",
)
SeqIO.write(mrna_record, fasta_mrna_output, "fasta")
print(f"\n[SALIDA] Archivo de ARNm guardado con éxito en: {fasta_mrna_output.name}")
print("=" * 75)

PASO 2: TRANSCRIPCIÓN DE ADN A ARN MENSAJERO (ARNm)
[MECANISMO MOLECULAR]
  * La ARN Polimerasa II reconoce el promotor y se acopla a la hebra molde (3'->5').
  * Incorpora NTPs complementarios (A->U, T->A, C->G, G->C) en sentido 5' -> 3'.

[SECUENCIA TRANSCRITA]
  > Molde de ADN leído (3'->5'):  3'- TACCGGGACACCTACGCGGAGGACGGGGAC... -5'
  > Transcrito de ARNm (5'->3'):  5'- AUGGCCCUGUGGAUGCGCCUCCUGCCCCUG... -3'
  > Longitud del ARNm:            333 ribonucleótidos

[SALIDA] Archivo de ARNm guardado con éxito en: ejercicio_06_insulina_arnm.fasta


### Paso 4: Traducción a Proteína Funcional y Arquitectura de Dominios

Simulamos la síntesis proteica por el complejo ribosómico:
1. Traducimos el ARNm a la secuencia de aminoácidos mediante `.translate(to_stop=True)`.
2. Validamos mediante `assert` que el polipéptido obtenido coincide exactamente al $100\%$ con la **Preproinsulina humana canónica del NCBI (`NP_000198.1`, 110 aa)**.
3. Desglosamos la proteína en sus **cuatro dominios biológicos** esenciales (Péptido señal, Cadena B, Péptido C y Cadena A).
4. Exportamos la proteína al archivo FASTA `data/ejercicio_06_insulina_proteina.fasta`.

In [5]:
# 3. Traducción de ARNm a Proteína
protein_seq = mrna_seq.translate(to_stop=True)

print("=" * 75)
print("PASO 3: TRADUCCIÓN A PROTEÍNA (PREPROINSULINA HUMANA)")
print("=" * 75)
print(f"Secuencia de aminoácidos: {protein_seq}")
print(f"Longitud total:           {len(protein_seq)} aminoácidos")
print(f"Extremo N-terminal:       {protein_seq[0]} (Metionina iniciadora)")
print(f"Extremo C-terminal:       {protein_seq[-1]} (Asparagina terminal)")

# Validación contra la secuencia canónica del NCBI NP_000198.1
ncbi_esperada_insulina = (
    "MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTR"
    "REAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN"
)
assert str(protein_seq) == ncbi_esperada_insulina, "Discrepancia con la secuencia de referencia NP_000198.1"
print("\n[VALIDACIÓN] ¡Concordancia exacta al 100% con la Preproinsulina humana (NP_000198.1)!")

# Desglose de los 4 dominios funcionales
peptido_senal = protein_seq[0:24]
cadena_b = protein_seq[24:54]
peptido_c = protein_seq[54:89]
cadena_a = protein_seq[89:110]

print("\n[ARQUITECTURA DE DOMINIOS DE LA PREPROINSULINA]")
print(f"  • Péptido Señal (aa 1-24):   {peptido_senal} (Translocación al RE)")
print(f"  • Cadena B (aa 25-54):       {cadena_b} (Hormona activa madura)")
print(f"  • Péptido C conector (55-89): {peptido_c} (Guía puentes disulfuro / escindido)")
print(f"  • Cadena A (aa 90-110):      {cadena_a} (Hormona activa madura con disulfuros)")

# Exportar archivo FASTA de Proteína
prot_record = SeqRecord(
    protein_seq,
    id="NP_000198.1_Insulina_Humana",
    description="Preproinsulina humana madura (110 aa) generada mediante pipeline Biopython",
)
SeqIO.write(prot_record, fasta_prot_output, "fasta")
print(f"\n[SALIDA] Archivo de Proteína guardado en: {fasta_prot_output.name}")
print("=" * 75)

PASO 3: TRADUCCIÓN A PROTEÍNA (PREPROINSULINA HUMANA)
Secuencia de aminoácidos: MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN
Longitud total:           110 aminoácidos
Extremo N-terminal:       M (Metionina iniciadora)
Extremo C-terminal:       N (Asparagina terminal)

[VALIDACIÓN] ¡Concordancia exacta al 100% con la Preproinsulina humana (NP_000198.1)!

[ARQUITECTURA DE DOMINIOS DE LA PREPROINSULINA]
  • Péptido Señal (aa 1-24):   MALWMRLLPLLALLALWGPDPAAA (Translocación al RE)
  • Cadena B (aa 25-54):       FVNQHLCGSHLVEALYLVCGERGFFYTPKT (Hormona activa madura)
  • Péptido C conector (55-89): RREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKR (Guía puentes disulfuro / escindido)
  • Cadena A (aa 90-110):      GIVEQCCTSICSLYQLENYCN (Hormona activa madura con disulfuros)

[SALIDA] Archivo de Proteína guardado en: ejercicio_06_insulina_proteina.fasta


### Paso 5: Implementación Orientada a Objetos — Clase `DogmaCentralPipeline`

Para estructurar el código de acuerdo a las mejores prácticas de ingeniería de software bioinformático, encapsulamos todo el flujo en la clase **`DogmaCentralPipeline`** implementada en `scripts/ejercicio_06_pipeline_dogma.py`.

Esta clase gestiona el estado, valida cada transición y emite alertas detalladas de cada paso en consola.

In [6]:
class DogmaCentralPipeline:
    """Orquestador bioinformático del Dogma Central con registro detallado de eventos."""

    def __init__(self, fasta_input: Path) -> None:
        self.fasta_input = fasta_input
        self.record_dna = None
        self.dna_seq_5_3 = None
        self.dna_template_3_5 = None
        self.mrna_seq_5_3 = None
        self.protein_seq = None

    def log_step(self, step_number: int, title: str) -> None:
        print("\n" + "=" * 75)
        print(f"PASO {step_number}: {title.upper()}")
        print("=" * 75)

    def ejecutar_flujo_completo(self, output_mrna: Path, output_prot: Path) -> None:
        # Paso 0: Carga
        self.log_step(0, "Carga de Secuencia FASTA")
        self.record_dna = SeqIO.read(self.fasta_input, "fasta")
        self.dna_seq_5_3 = self.record_dna.seq
        print(f"  [OK] Gen: {self.record_dna.id} ({len(self.dna_seq_5_3)} pb cargados)")

        # Paso 1: Replicación
        self.log_step(1, "Replicación Semiconservativa")
        self.dna_template_3_5 = self.dna_seq_5_3.complement()
        print(f"  [OK] Hebra parental (5'->3'):   5'- {self.dna_seq_5_3[:25]}... -3'")
        print(f"  [OK] Hebra molde nueva (3'->5'): 3'- {self.dna_template_3_5[:25]}... -5'")

        # Paso 2: Transcripción
        self.log_step(2, "Transcripción a ARNm")
        self.mrna_seq_5_3 = self.dna_seq_5_3.transcribe()
        print(f"  [OK] Transcrito primario (5'->3'): 5'- {self.mrna_seq_5_3[:25]}... -3'")
        mrna_rec = SeqRecord(self.mrna_seq_5_3, id=f"{self.record_dna.id}_mRNA", description="ARNm maduro")
        SeqIO.write(mrna_rec, output_mrna, "fasta")
        print(f"  [SALIDA] Guardado en: {output_mrna.name}")

        # Paso 3: Traducción
        self.log_step(3, "Traducción a Proteína")
        self.protein_seq = self.mrna_seq_5_3.translate(to_stop=True)
        print(f"  [OK] Proteína traducida ({len(self.protein_seq)} aa): {self.protein_seq[:30]}...")
        prot_rec = SeqRecord(self.protein_seq, id=f"{self.record_dna.id}_PROT", description="Proteína funcional")
        SeqIO.write(prot_rec, output_prot, "fasta")
        print(f"  [SALIDA] Guardado en: {output_prot.name}")
        print("\n>> PIPELINE COMPLETADO EXITOSAMENTE: ADN -> ARN -> PROTEÍNA")

# Instanciación y ejecución del pipeline completo
pipeline = DogmaCentralPipeline(fasta_dna_input)
pipeline.ejecutar_flujo_completo(fasta_mrna_output, fasta_prot_output)


PASO 0: CARGA DE SECUENCIA FASTA
  [OK] Gen: lcl|NM_000207.3_cds_NP_000198.1_1 (333 pb cargados)

PASO 1: REPLICACIÓN SEMICONSERVATIVA
  [OK] Hebra parental (5'->3'):   5'- ATGGCCCTGTGGATGCGCCTCCTGC... -3'
  [OK] Hebra molde nueva (3'->5'): 3'- TACCGGGACACCTACGCGGAGGACG... -5'

PASO 2: TRANSCRIPCIÓN A ARNM
  [OK] Transcrito primario (5'->3'): 5'- AUGGCCCUGUGGAUGCGCCUCCUGC... -3'
  [SALIDA] Guardado en: ejercicio_06_insulina_arnm.fasta

PASO 3: TRADUCCIÓN A PROTEÍNA
  [OK] Proteína traducida (110 aa): MALWMRLLPLLALLALWGPDPAAAFVNQHL...
  [SALIDA] Guardado en: ejercicio_06_insulina_proteina.fasta

>> PIPELINE COMPLETADO EXITOSAMENTE: ADN -> ARN -> PROTEÍNA


### Paso 6 (Extra interactivo): Pipeline Universal para Cualquier Secuencia de ADN

Para permitir al usuario ingresar cualquier secuencia personalizada de ADN y observar en tiempo real la replicación, transcripción y traducción completas, definimos la función `pipeline_dogma_universal(secuencia_dna, nombre)`:

In [7]:
def pipeline_dogma_universal(secuencia_dna_str: str, nombre_gen: str = "Gen_Custom") -> dict:
    """
    Ejecuta el Dogma Central completo sobre cualquier secuencia de ADN codificante dada:
    Replicación -> Transcripción -> Traducción.
    """
    limpia = secuencia_dna_str.strip().upper()
    dna = Seq(limpia)
    
    # 1. Replicación
    molde_3_5 = dna.complement()
    rev_comp_5_3 = dna.reverse_complement()
    
    # 2. Transcripción
    mrna = dna.transcribe()
    
    # 3. Traducción
    proteina = mrna.translate(to_stop=True)
    
    print(f"\n>>> PIPELINE UNIVERSAL PARA: {nombre_gen}")
    print(f"    ADN parental (5'->3'):   5'- {dna} -3'")
    print(f"    Molde replicado (3'->5'):3'- {molde_3_5} -5'")
    print(f"    ARNm transcrito (5'->3'):5'- {mrna} -3'")
    print(f"    Proteína traducida:      {proteina} ({len(proteina)} aa)")
    
    return {"dna": dna, "molde": molde_3_5, "mrna": mrna, "proteina": proteina}

# Prueba con una secuencia corta de ejemplo con codón de parada:
res = pipeline_dogma_universal("ATGCCCGAAACTTAA", "Peptido_Test")


>>> PIPELINE UNIVERSAL PARA: Peptido_Test
    ADN parental (5'->3'):   5'- ATGCCCGAAACTTAA -3'
    Molde replicado (3'->5'):3'- TACGGGCTTTGAATT -5'
    ARNm transcrito (5'->3'):5'- AUGCCCGAAACUUAA -3'
    Proteína traducida:      MPET (4 aa)



## 📌 Resumen de lo aprendido

- **Flujo colineal de la información génica:** El ADN bicatenario actúa como depósito de memoria; la ARN polimerasa transcribe el mensaje monocatenario a ARNm y el ribosoma decodifica los tripletes en la secuencia primaria de aminoácidos.
- **Integración con Biopython:** Los métodos `.complement()`, `.transcribe()` y `.translate(to_stop=True)` modelan con precisión matemática y biológica las tres etapas centrales.
- **Vulnerabilidad a errores:** Aunque la traducción presenta la mayor tasa de fallos por aminoácido (errores transitorios y diluidos por el recambio proteico), **la Replicación del ADN es el eslabón más vulnerable** del dogma central, pues las mutaciones en el ADN son indelebles, se fijan covalentemente y se multiplican en cascada en todos los ARNm y proteínas celulares descendientes.